### data processing bsub

In [ ]:
bsub -J ALL -q standard\
  -n 4 -R "span[hosts=1]" \
  -R "rusage[mem=40G]" \
  -o %J.out -e %J.err \
  bash -lc '
module load conda3 
source activate torch_gpu

export OMP_NUM_THREADS=8 MKL_NUM_THREADS=8

python dataProcessing_accel.py \
  --fasta ./hg38/hg38.fa \
  --cells cells_atac.tsv \
  --tracks tracks.tsv \
  --out_dir ./dex_atac/datasets_atac \
  --sequence_length 2048 --target_length 1024 --bin_size 2 \
  --val_chroms chr8 chr14 \
  --test_chroms chr10 chr22 chr4\
  --allow_regex "^chr([1-9]|1[0-9]|2[0-2]|X|Y)$" \
  --blacklist ./hg38/hg38-blacklist.v2.bed \
  --num_workers 8 \
  --shard_size 2048 \
  --batch_bw_per_chrom \
  --max_batch_bp 5000000 \
  --min_peak_distance_bp 1024
'

In [ ]:
bsub -J ALL0 -q standard\
  -n 4 -R "span[hosts=1]" \
  -R "rusage[mem=40G]" \
  -o %J.out -e %J.err \
  bash -lc '
module load conda3 
source activate torch_gpu

export OMP_NUM_THREADS=8 MKL_NUM_THREADS=8

python dataProcessing_accel.py \
  --fasta ./hg38/hg38.fa \
  --cells cells_atac2.tsv \
  --tracks tracks2.tsv \
  --out_dir ./dex_atac/datasets_atac2 \
  --sequence_length 2048 --target_length 1024 --bin_size 2 \
  --val_chroms chr8 chr14 \
  --test_chroms chr10 chr22 chr4\
  --allow_regex "^chr([1-9]|1[0-9]|2[0-2]|X|Y)$" \
  --blacklist ./hg38/hg38-blacklist.v2.bed \
  --num_workers 8 \
  --shard_size 2048 \
  --batch_bw_per_chrom \
  --max_batch_bp 5000000 \
  --min_peak_distance_bp 1024
'

In [ ]:
############## GR_LEF1 labeled peak
bsub -J cell_up -q standard\
  -n 4 -R "span[hosts=1]" \
  -R "rusage[mem=40G]" \
  -o %J.out -e %J.err \
  bash -lc '
module load conda3 
source activate torch_gpu

export OMP_NUM_THREADS=8 MKL_NUM_THREADS=8

python dataProcessing.py \
  --fasta ./hg38/hg38.fa \
  --cells cells_up.tsv \
  --tracks tracks.tsv \
  --out_dir ./peaks_dexatac/peaks_both/up \
  --sequence_length 2048 --target_length 1024 --bin_size 2 \
  --allow_regex "^chr([1-9]|1[0-9]|2[0-2]|X|Y)$" \
  --blacklist ./hg38/hg38-blacklist.v2.bed \
  --min_peak_distance_bp 1024
'
##################################

bsub -J cell2_up -q standard\
  -n 4 -R "span[hosts=1]" \
  -R "rusage[mem=40G]" \
  -o %J.out -e %J.err \
  bash -lc '
module load conda3 
source activate torch_gpu

export OMP_NUM_THREADS=8 MKL_NUM_THREADS=8

python dataProcessing.py \
  --fasta ./hg38/hg38.fa \
  --cells cells2_up.tsv \
  --tracks tracks2.tsv \
  --out_dir ./peaks_dexatac/peaks2_both/up \
  --sequence_length 2048 --target_length 1024 --bin_size 2 \
  --allow_regex "^chr([1-9]|1[0-9]|2[0-2]|X|Y)$" \
  --blacklist ./hg38/hg38-blacklist.v2.bed \
  --min_peak_distance_bp 1024
'

### train

In [ ]:
bsub -J RS411_2 -q dgx \
  -n 1 -R "span[hosts=1]" \
  -R "rusage[mem=200G]"  \
  -gpu "num=1:j_exclusive=yes:mode=exclusive_process" \
  -o %J.out -e %J.err \
  bash -lc '
module load conda3
source activate torch_gpu
export CUDA_VISIBLE_DEVICES=0
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
# quick sanity
python -V
python -c "import torch; print(\"torch\", torch.__version__, \"cuda:\", torch.cuda.is_available(), \"gpus:\", torch.cuda.device_count())"

python train_eval.py \
  --mode train \
  --out_dir runs/datasets2/RS411 \
  --train_list datasets2/RS411/train_files.txt \
  --val_list datasets2/RS411/val_files.txt \
  --heads "dex=1, dmso=1" \
  --model_type motif-based-model \
  --motif_use_prior \
  --motif_pwm_path ./meme/consensus_pwms.meme \
  --sequence_length 2048 \
  --target_length 1024 \
  --batch 8 \
  --epochs 20 \
  --auto_fit_geometry \
  --lr 1e-5 \
  --amp
'

In [ ]:
bsub -J RS411 -q dgx \
  -n 1 -R "span[hosts=1]" \
  -R "rusage[mem=200G]"  \
  -gpu "num=1:j_exclusive=yes:mode=exclusive_process" \
  -o %J.out -e %J.err \
  bash -lc '
module load conda3
source activate torch_gpu
export CUDA_VISIBLE_DEVICES=0
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
# quick sanity
python -V
python -c "import torch; print(\"torch\", torch.__version__, \"cuda:\", torch.cuda.is_available(), \"gpus:\", torch.cuda.device_count())"

python train_eval.py \
  --mode train \
  --out_dir runs/2atac/datasets_atac/RS411 \
  --train_list 2atac/datasets_atac/RS411/train_files.txt \
  --val_list 2atac/datasets_atac/RS411/val_files.txt \
  --heads "rs411=1" \
  --model_type motif-based-model \
  --motif_use_prior \
  --motif_pwm_path ./meme/consensus_pwms.meme \
  --sequence_length 2048 \
  --target_length 1024 \
  --batch 4 \
  --epochs 30 \
  --auto_fit_geometry \
  --lr 1e-5 \
  --amp
'

### test

In [ ]:
bsub -J TSUPB15 -q dgx \
  -n 1 -R "span[hosts=1]" \
  -R "rusage[mem=200G]"  \
  -gpu "num=1:j_exclusive=yes:mode=exclusive_process" \
  -o %J.out -e %J.err \
  bash -lc '
module load conda3
source activate torch_gpu
export CUDA_VISIBLE_DEVICES=0
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
# quick sanity
python -V
python -c "import torch; print(\"torch\", torch.__version__, \"cuda:\", torch.cuda.is_available(), \"gpus:\", torch.cuda.device_count())"

python train_eval.py \
  --mode test \
  --motif_use_prior \
  --motif_pwm_path meme/consensus_pwms.meme \
  --test_list datasets/SUPB15/test_files.txt \
  --out_dir runs/datasets/SUPB15 \
  --ckpt runs/datasets/SUPB15/best.pt \
  --heads "supb15=1" \
  --batch 8 \
  --amp
'

## Combined bash codes

### Train and test

In [ ]:
bsub -J RS411 -q dgx \
  -n 1 -R "span[hosts=1]" \
  -R "rusage[mem=200G]" \
  -gpu "num=1:j_exclusive=yes:mode=exclusive_process" \
  -o %J.out -e %J.err \
  bash -lc '
set -euo pipefail

# ====== change only this ======
CELL="RS411"
# =============================

module load conda3

export MKL_INTERFACE_LAYER="${MKL_INTERFACE_LAYER:-LP64}"
export MKL_THREADING_LAYER="${MKL_THREADING_LAYER:-INTEL}"

source activate torch_gpu

export CUDA_VISIBLE_DEVICES=0
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

# quick sanity
python -V
python -c "import torch; print(\"torch\", torch.__version__, \"cuda:\", torch.cuda.is_available(), \"gpus:\", torch.cuda.device_count())"

# ====== consistent paths ======
DATA_DIR="dex_atac/datasets_atac/${CELL}"
RUN_DIR="runs/dex_atac/datasets_atac/${CELL}"

PWM_PATH="meme/consensus_pwms.meme"
MODEL_TYPE="motif-based-model"

HEADS="rs411=1"
BATCH=8
# =============================

mkdir -p "${RUN_DIR}"

echo "=== [1/2] TRAIN: ${CELL} ==="
python train_eval.py \
  --mode train \
  --out_dir "${RUN_DIR}" \
  --train_list "${DATA_DIR}/train_files.txt" \
  --val_list "${DATA_DIR}/val_files.txt" \
  --heads "${HEADS}" \
  --model_type "${MODEL_TYPE}" \
  --motif_use_prior \
  --motif_pwm_path "${PWM_PATH}" \
  --sequence_length 2048 \
  --target_length 1024 \
  --batch "${BATCH}" \
  --epochs 20 \
  --auto_fit_geometry \
  --lr 1e-5 \
  --amp

CKPT="${RUN_DIR}/best.pt"
if [[ ! -f "${CKPT}" ]]; then
  echo "ERROR: checkpoint not found: ${CKPT}"
  exit 1
fi

echo "=== [2/2] TEST: ${CELL} ==="
python train_eval.py \
  --mode test \
  --out_dir "${RUN_DIR}" \
  --test_list "${DATA_DIR}/test_files.txt" \
  --ckpt "${CKPT}" \
  --heads "${HEADS}" \
  --model_type "${MODEL_TYPE}" \
  --motif_use_prior \
  --motif_pwm_path "${PWM_PATH}" \
  --batch "${BATCH}" \
  --amp

echo "ALL DONE for ${CELL}"
'

In [ ]:
#### Train and test
bsub -J RS411_2 -q dgx \
  -n 1 -R "span[hosts=1]" \
  -R "rusage[mem=200G]" \
  -gpu "num=1:j_exclusive=yes:mode=exclusive_process" \
  -o %J.out -e %J.err \
  bash -lc '
set -euo pipefail

# ====== change only this ======
CELL="RS411"
# =============================

module load conda3

export MKL_INTERFACE_LAYER="${MKL_INTERFACE_LAYER:-LP64}"
export MKL_THREADING_LAYER="${MKL_THREADING_LAYER:-INTEL}"

source activate torch_gpu

export CUDA_VISIBLE_DEVICES=0
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

# quick sanity
python -V
python -c "import torch; print(\"torch\", torch.__version__, \"cuda:\", torch.cuda.is_available(), \"gpus:\", torch.cuda.device_count())"

# ====== consistent paths ======
DATA_DIR="dex_atac/datasets_atac2/${CELL}"
RUN_DIR="runs/dex_atac/datasets_atac2/${CELL}"

PWM_PATH="meme/consensus_pwms.meme"
MODEL_TYPE="motif-based-model"

HEADS="dex=1, dmso=1"
BATCH=8
# =============================

mkdir -p "${RUN_DIR}"

echo "=== [1/2] TRAIN: ${CELL} ==="
python train_eval.py \
  --mode train \
  --out_dir "${RUN_DIR}" \
  --train_list "${DATA_DIR}/train_files.txt" \
  --val_list "${DATA_DIR}/val_files.txt" \
  --heads "${HEADS}" \
  --model_type "${MODEL_TYPE}" \
  --motif_use_prior \
  --motif_pwm_path "${PWM_PATH}" \
  --sequence_length 2048 \
  --target_length 1024 \
  --batch "${BATCH}" \
  --epochs 20 \
  --auto_fit_geometry \
  --lr 1e-5 \
  --amp

CKPT="${RUN_DIR}/best.pt"
if [[ ! -f "${CKPT}" ]]; then
  echo "ERROR: checkpoint not found: ${CKPT}"
  exit 1
fi

echo "=== [2/2] TEST: ${CELL} ==="
python train_eval.py \
  --mode test \
  --out_dir "${RUN_DIR}" \
  --test_list "${DATA_DIR}/test_files.txt" \
  --ckpt "${CKPT}" \
  --heads "${HEADS}" \
  --model_type "${MODEL_TYPE}" \
  --motif_use_prior \
  --motif_pwm_path "${PWM_PATH}" \
  --batch "${BATCH}" \
  --amp

echo "ALL DONE for ${CELL}"
'